In [ ]:
"""
https://freesound.org/search/?q=&f=tag%253A%2522field-recording%2522&s=Duration+%28longest+first%29&si_tags=0&si_name=0&si_description=0&si_packname=0&si_sound_id=0&si_username=0&d0=0&d1=*&ig=0&r=0&g=1&dp=0&cm=0&mm=0
field-recordings

Research Questions:
 - Question 1 : Can we detect when there is a strange noise in the audio (e.g a drone in a forest, a car in the ocean)
    Can we identify the unnatural sound segment even when overlapped with audio from the base env.
    - Method : Probably some unsupervised modal  (GMM? )

 - Question 2 : Can we classify those noises accurately
    - Method : Probably a DNN to learn what a drone, plane, etc sounds like
 
 - Question 3 : Can we determine additional information regarding position of the anomaly
    - Method : Kalman Filter : Model the relationship of distance and noise , and filter noise

"""



In [ ]:
"""
Audio Data (keep learning about the audio data struct)

- Mel Frequencies: 
Mel frequencies are a way of representing sound frequencies according to how humans 
percieve pitch. Humans do not percieve pitch linearly. The mel scale transforms 
frquencies (Hz) into a scale that is approx linear to how its percieved.

- Hz Frequencies: 
How many times a wave occurs in a second (peak to peak, zero to zero, or trough to trough)

- The y axis (the vals in the array are the amplitude), x axis is time

"""

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
#Shortening all base audio to max 1hr

import soundfile as sf
import gc
from pathlib import Path

start_sec = 0
duration = 60 * 60  # 1 hour max
CHUNK_FRAMES = 441000  # 10 seconds at 44100hz

source_to_dest = {
    "./Data/Audio_Files/Raw_Audios/BaseAudios/NonUrban":
        r"C:\Users\rubie\Desktop\RFCUNY Research\Data\Audio_Files\Raw_Audios\Shortened_Base_Audios\NonUrban",
    "./Data/Audio_Files/Raw_Audios/BaseAudios/Urban":
        r"C:\Users\rubie\Desktop\RFCUNY Research\Data\Audio_Files\Raw_Audios\Shortened_Base_Audios\Urban",
}

for source_path, dest_path in source_to_dest.items():

    source_dir = Path(source_path)
    dest_dir = Path(dest_path)
    dest_dir.mkdir(parents=True, exist_ok=True)

    files = [f for f in source_dir.glob("*")]
    total = len(files)
    counter = 0

    for audio in files:
        print(f"Processing: {audio.name}")  # see exactly which file crashes
        with sf.SoundFile(audio) as f:
            sr = f.samplerate
            start_frame = int(start_sec * sr)
            max_frames = int(duration * sr)
            f.seek(start_frame)
            frames_remaining = max_frames

            output_path = dest_dir / audio.name
            with sf.SoundFile(output_path, mode='w', samplerate=sr, channels=f.channels) as out:
                while frames_remaining > 0:
                    chunk = f.read(frames=min(CHUNK_FRAMES, frames_remaining), dtype='float32', always_2d=False)
                    if len(chunk) == 0:
                        break
                    out.write(chunk)
                    frames_remaining -= len(chunk)
                    del chunk

        gc.collect()  # once per file, not per chunk
        counter += 1
        print(f"Progress: {counter / total:.2%}")

    print(f"Dir complete: {source_path}")

Processing: 102934__lg__waterfalls-in-saas-fee-def.wav
Progress: 3.85%
Processing: 120905__kyster__the-morning-comes-20-5-11.wav
Progress: 7.69%
Processing: 172522__klankbeeld__strong-wind-river-side-121226_03.flac
Progress: 11.54%
Processing: 192253__csengeri__23rdjune2013thunderstorm.mp3
Progress: 15.38%
Processing: 218515__daveincamas__20130922stormoregoncoast.flac
Progress: 19.23%
Processing: 235447__csengeri__easter-morning-2014.mp3
Progress: 23.08%
Processing: 249411__csengeri__14th-september-2014-rainfall.mp3
Progress: 26.92%
Processing: 276268__csengeri__150609005jegeso.mp3
Progress: 30.77%
Processing: 348120__marlo1981__summer-night-storm-at-the-country-side.ogg
Progress: 34.62%
Processing: 348510__marlo1981__country-side-night-ambience.mp3
Progress: 38.46%
Processing: 350666__csengeri__23rd-july-2016-thunderstorm-dictaphone-was-under-a-tree-branch.mp3
Progress: 42.31%
Processing: 400791__rizzin__korean-island-shore-at-night.ogg
Progress: 46.15%
Processing: 435674__csengeri__2

In [2]:
import librosa
import numpy as np

In [3]:
y, sr = librosa.load("./Data/Audio_Files/Raw_Audios/BaseAudios/NonUrban/348120__marlo1981__summer-night-storm-at-the-country-side.ogg")

# Librosa samples at 22050 Hz by default
# mfccs = librosa.feature.mfcc(y=y, sr=sr)

In [3]:
from Generate_AudioData import DJ_splice

dj_splice = DJ_splice()

2026-06-19 06:49:05,715 - INFO - Audio file table of content has been updated


In [5]:
#Shorten the clip for testing
audio_len = (15*60) #15 minutes in seconds
short_y = dj_splice._ClipAudio(y, sr, start_sec=0, duration_sec=audio_len)

In [15]:
#Testing the Create_Audio_Data method
original_audio = short_y
anomaly_type = "Gunshots" 
sample_rate = sr 
overlayed_2_original_ratio = .3 
segment_duration = 10

result = dj_splice.Create_Audio_Data(original_audio, anomaly_type, sample_rate, overlayed_2_original_ratio, segment_duration)




In [16]:
result[0] #Audio segments
result[1] #Segments that were overlayed with gunshot noises

[(np.int32(37), np.float64(0.40085039230648084)),
 (np.int32(57), np.float64(0.0555140137642647)),
 (np.int32(53), np.float64(0.3174143257084652)),
 (np.int32(7), np.float64(0.28392262342336955)),
 (np.int32(38), np.float64(0.03430169844868949)),
 (np.int32(6), np.float64(0.47621285336452807)),
 (np.int32(33), np.float64(0.06872466072849755)),
 (np.int32(20), np.float64(0.24593814844696937)),
 (np.int32(15), np.float64(0.463757992444639)),
 (np.int32(45), np.float64(0.3337844788800243)),
 (np.int32(1), np.float64(0.04132548867570778)),
 (np.int32(27), np.float64(0.07866471580344224)),
 (np.int32(43), np.float64(0.23690227215023568)),
 (np.int32(68), np.float64(0.13846011623085056)),
 (np.int32(76), np.float64(0.11287247509128084)),
 (np.int32(74), np.float64(0.03322416526076938)),
 (np.int32(9), np.float64(0.3598026717428237)),
 (np.int32(28), np.float64(0.40124226613377606)),
 (np.int32(21), np.float64(0.2886648231950596)),
 (np.int32(77), np.float64(0.04400663417522201)),
 (np.int32(

In [19]:
from IPython.display import Audio

Audio(result[0][7], rate=sr)

In [ ]:
#Note we are forcing the audio into a single channel by default librosa mono=True

In [ ]:
# 30% of the base audio segments are overlayed with anomalies
# 10 sec segments
# Loudness scaling of anomaly random var uniform dist [.01, .5]

In [4]:
#Get Data Prepped
from pathlib import Path
import librosa
import numpy as np

#Using Shortened Base Audios
toc_file_path = Path("./Data/Audio_Files/Clips/non_urban_toc.json") 
Urban_nonUbran_bool = 0 #0 non urban, 1 urban 
anomaly_list = [
    "Animals",
    "Explosion",
    "Gunshots",
    "Human_Voices",
    "UAV"
]
overlayed_2_original_ratio = .3 #30% 
segment_duration = 10 #10 sec segments (clips)

In [5]:
result = dj_splice.PrepareData(toc_file_path, Urban_nonUbran_bool, anomaly_list, overlayed_2_original_ratio, segment_duration)

2026-06-18 21:10:35,031 - INFO - Files will be saved to the NonUrban Directory


2026-06-18 21:12:16,885 - INFO - Base audio processed: 1/26
2026-06-18 21:13:28,512 - INFO - Base audio processed: 1/26
2026-06-18 21:14:24,976 - INFO - Base audio processed: 1/26
2026-06-18 21:15:34,870 - INFO - Base audio processed: 1/26
2026-06-18 21:16:33,550 - INFO - Base audio processed: 1/26
2026-06-18 21:17:47,543 - INFO - Base audio processed: 1/26
2026-06-18 21:18:59,380 - INFO - Base audio processed: 1/26
2026-06-18 21:20:16,646 - INFO - Base audio processed: 1/26
2026-06-18 21:21:29,295 - INFO - Base audio processed: 1/26
2026-06-18 21:22:43,617 - INFO - Base audio processed: 1/26
2026-06-18 21:24:03,923 - INFO - Base audio processed: 1/26
2026-06-18 21:25:20,979 - INFO - Base audio processed: 1/26
2026-06-18 21:26:34,143 - INFO - Base audio processed: 1/26
2026-06-18 21:27:52,785 - INFO - Base audio processed: 1/26
2026-06-18 21:29:03,983 - INFO - Base audio processed: 1/26
2026-06-18 21:30:17,497 - INFO - Base audio processed: 1/26
2026-06-18 21:31:27,821 - INFO - Base au

In [4]:
#Run Data Prep for urban 
#Get Data Prepped
from pathlib import Path
import librosa
import numpy as np

#Using Shortened Base Audios
toc_file_path = Path("./Data/Audio_Files/Clips/urban_toc.json") 
Urban_nonUbran_bool = 1 #0 non urban, 1 urban 
anomaly_list = [
    "Animals",
    "Explosion",
    "Gunshots",
    "Human_Voices",
    "UAV"
]
overlayed_2_original_ratio = .3 #30% 
segment_duration = 10 #10 sec segments (clips)

In [5]:
result = dj_splice.PrepareData(toc_file_path, Urban_nonUbran_bool, anomaly_list, overlayed_2_original_ratio, segment_duration)

2026-06-19 06:49:56,753 - INFO - Files will be saved to the Urban Directory
2026-06-19 06:51:14,571 - INFO - Base audio processed: 1/30
2026-06-19 06:52:06,500 - INFO - Base audio processed: 2/30
2026-06-19 06:52:43,127 - INFO - Base audio processed: 3/30
2026-06-19 06:53:59,664 - INFO - Base audio processed: 4/30
2026-06-19 06:55:08,500 - INFO - Base audio processed: 5/30
2026-06-19 06:55:46,869 - INFO - Base audio processed: 6/30
2026-06-19 06:56:13,550 - INFO - Base audio processed: 7/30
2026-06-19 06:56:51,338 - INFO - Base audio processed: 8/30
2026-06-19 06:57:19,849 - INFO - Base audio processed: 9/30
2026-06-19 06:58:34,965 - INFO - Base audio processed: 10/30
2026-06-19 06:59:03,020 - INFO - Base audio processed: 11/30
2026-06-19 07:00:01,627 - INFO - Base audio processed: 12/30
2026-06-19 07:00:33,651 - INFO - Base audio processed: 13/30
2026-06-19 07:00:56,458 - INFO - Base audio processed: 14/30
2026-06-19 07:01:34,331 - INFO - Base audio processed: 15/30
2026-06-19 07:02:5

In [ ]:
# JSON structure:
# Assumes Monotone= True
# {
#     "base_audio_file_name": "",
#     "segment_durations": , #In secs
#     "anomaly_to_original_ratio": ,
#     "anomaly_info": [
#         # (anomaly_segment_index, scaling_factor),
#         # ...
#     ],
#     "anomaly_type": "",
#     "sample_rate": , 
# }